# Notebook 1 — 100-example test run

Runs **up to 100 examples** of the corrected thesis pipeline to confirm the prediction/scoring path is healthy before the full experiment.

**Safeguards**
- Never prints API keys.
- Checkpointed & resumable: completed successful cases are not re-run.
- API/parse failures are logged separately and are **not** counted as wrong answers.
- Reproducible sampling via a fixed `GLOBAL_SEED`.

Run all cells top to bottom.

In [ ]:
# --- Environment & configuration -------------------------------------------
import os, sys, json, importlib
from pathlib import Path
import pandas as pd

# Make the project importable whether the kernel starts in notebooks/ or root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
for _m in [m for m in list(sys.modules) if m == "thesis_pipeline" or m.startswith("thesis_pipeline.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# TLS: the SDKs verify certificates by default. Behind a TLS-intercepting
# corporate proxy every API call raises APIConnectionError and the pipeline
# would record empty predictions. Setting VERIFY_SSL=false in .env (or here)
# restores connectivity. Leave it "true" on a normal network.
os.environ.setdefault("VERIFY_SSL", os.environ.get("VERIFY_SSL", "true"))

from thesis_pipeline import initialize_thesis
from thesis_pipeline.evaluation import score, metrics
from thesis_pipeline.checkpoint import load_completed

# Report key presence ONLY (never print the key values themselves).
print("OPENAI_API_KEY set:  ", bool(os.environ.get("OPENAI_API_KEY")))
print("ANTHROPIC_API_KEY set:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("VERIFY_SSL:          ", os.environ.get("VERIFY_SSL"))

In [ ]:
# --- Dataset & pipeline ----------------------------------------------------
# Change only this cell to point at a different CSV/JSON/JSONL dataset.
DATASET = {
    "name": "final_dataset_1",
    # Reliable extracted gold letters (built by tools/build_gold_labels.py).
    "path": "data/final_dataset_1_with_gold.csv",
    "columns": {"instruction": "instruction", "input": "input", "gold": "gold_letter"},
}
GLOBAL_SEED = 42

# Reproducible sample size. THESIS_MAX_EXAMPLES lets a smoke test override it.
MAX_EXAMPLES = int(os.environ.get("THESIS_MAX_EXAMPLES", "100"))

pipeline = initialize_thesis(
    DATASET, RUN_MODE="test", GLOBAL_SEED=GLOBAL_SEED, root=ROOT,
    mock=False, test_size=MAX_EXAMPLES, output_dir="results/run_100_examples",
)
v = pipeline.bundle.validation
print("Dataset:", pipeline.bundle.dataset_name)
print(f"Total rows: {len(pipeline.bundle.frame)} | eligible (clean A-H gold): {v['eligible_cases']} | excluded: {v['excluded_cases']}")
print("Excluded-case audit written to:", pipeline.output / "excluded_cases.csv")
print("Class distribution (gold):", v["class_distribution"])
print(f"Development cases: {len(pipeline.bundle.cases('development'))} | Test cases: {len(pipeline.bundle.cases('test'))}")

In [ ]:
# --- Select up to MAX_EXAMPLES test cases (reproducible) --------------------
# The split is a deterministic function of GLOBAL_SEED, and .head() preserves a
# fixed order, so the same rows are chosen on every run.
cases = pipeline._cases("test")
print(f"Selected {len(cases)} test cases (seed={GLOBAL_SEED}).")
cases[["case_id", "gold"]].head()

In [ ]:
# --- Run the experiment (checkpointed / resumable) -------------------------
# Uses the corrected single-agent pipeline. Each provider writes a per-case
# JSONL checkpoint; re-running the cell resumes and does NOT repeat completed,
# successful cases. A failing case (e.g. bad key) is recorded with a status and
# error and is retried on the next run.
#
# THESIS_PROVIDERS can restrict which systems run (e.g. "single_gpt").
_providers = {
    "single_gpt":    ("openai",    pipeline.config.gpt_model),
    "single_claude": ("anthropic", pipeline.config.claude_model),
}
_wanted = os.environ.get("THESIS_PROVIDERS")
if _wanted:
    _providers = {k: v for k, v in _providers.items() if k in set(_wanted.split(","))}

predictions = {}
for name, (provider, model) in _providers.items():
    print(f"Running {name} ({provider}:{model}) on {len(cases)} cases ...")
    pred = pipeline._single_predictions(cases, name, provider, model, "test")
    predictions[name] = pred
    ok = (pred["status"] == "success").sum() if "status" in pred else 0
    print(f"  -> {ok}/{len(pred)} successful responses; checkpoint: {pipeline.output / (name + '_test.jsonl')}")

In [ ]:
# --- Save predictions, raw checkpoints, and errors (separately) ------------
scored = {}
for name, pred in predictions.items():
    s = score(cases, pred)
    scored[name] = s
    # Individual predictions (with correctness + status).
    out = s.merge(pred.drop(columns=[c for c in ["prediction","critical_safety_error"] if c in pred.columns]),
                  on="case_id", how="left", suffixes=("", "_raw"))
    out.to_csv(pipeline.output / f"{name}_predictions.csv", index=False)
    # Errors saved separately so API/parse failures are never confused with
    # genuine wrong answers.
    errs = pred[pred["status"] != "success"] if "status" in pred else pred.iloc[0:0]
    errs.to_csv(pipeline.output / f"{name}_errors.csv", index=False)
    print(f"{name}: predictions -> {name}_predictions.csv | errors ({len(errs)}) -> {name}_errors.csv")
    print(f"        raw per-case checkpoint (JSONL) -> {name}_test.jsonl")

In [ ]:
# --- Metrics & summary -----------------------------------------------------
rows = []
for name, s in scored.items():
    m = metrics(s, name)
    rows.append(m)
summary = pd.DataFrame(rows)[[
    "system", "n", "successful_n", "failure_count", "failure_rate",
    "accuracy", "accuracy_successful_only", "precision", "recall", "macro_f1",
    "critical_safety_error_rate",
]]
summary.to_csv(pipeline.output / "summary_metrics.csv", index=False)
summary

In [ ]:
# --- Clear status report ---------------------------------------------------
for name, pred in predictions.items():
    s = scored[name]
    n = len(pred)
    st = pred["status"] if "status" in pred else pd.Series(["success"] * n)
    successful = int((st == "success").sum())
    parse_fail = int((st == "parse_error").sum())
    other_fail = n - successful - parse_fail
    correct = int(s["correct"].sum())
    print(f"=== {name} ===")
    print(f"  total attempted:            {n}")
    print(f"  successful model responses: {successful}")
    print(f"  parsing failures:           {parse_fail}")
    print(f"  other failures (API/etc.):  {other_fail}")
    print(f"  correct predictions:        {correct}")
    print(f"  incorrect predictions:      {successful - correct}  (among successful only)")
    m = metrics(s, name)
    print(f"  accuracy (all attempted):   {m['accuracy']:.4f}")
    print(f"  accuracy (successful only): {m['accuracy_successful_only']}")
    print(f"  macro F1:                   {m['macro_f1']:.4f}\n")

### A few individual predictions

In [ ]:
# --- Inspect several individual predictions --------------------------------
name = next(iter(scored))
s = scored[name]
show = cases[["case_id", "instruction", "gold"]].merge(
    s[["case_id", "prediction", "correct"]], on="case_id", how="left").head(8)
for _, r in show.iterrows():
    stem = str(r["instruction"]).splitlines()[1] if len(str(r["instruction"]).splitlines()) > 1 else str(r["instruction"])[:90]
    print(f"[{str(r['case_id'])[:10]}] expected={r['gold']} predicted={r['prediction']} correct={bool(r['correct'])}")
    print(f"    {stem[:100]}")

If `accuracy_successful_only` is a real number that varies across systems (not all zero) and `failure_count` is low, the pipeline is healthy and you can proceed to **02_RUN_FULL_EXPERIMENT.ipynb**. If a system shows `failure_rate = 1.0`, fix its API key/model before continuing — that is an infrastructure error, not a model score of zero.